# Ultralytics YOLOv8 Training Notebook

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # go one level up

from utils import normalizer, reset, validation

In [ ]:
reset.delete_all_jpg_files("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData")

In [ ]:
rfc: str = "data/IMG00425.JPG"
mtd: normalizer.TransferMethod="mean_std"

In [ ]:
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images","data/BulkNormalizedAnnotatedData/images",transfer_method=mtd)
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images/train","data/BulkNormalizedAnnotatedData/images/train",transfer_method=mtd)
normalizer.batch_color_transfer(rfc,"data/BulkAnnotatedData/images/val","data/BulkNormalizedAnnotatedData/images/val",transfer_method=mtd)

In [ ]:
from ultralytics import YOLO
import optuna
import time
import os

In [ ]:
from ultralytics.data.utils import check_det_dataset

check_det_dataset("/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData/data.yaml")

In [ ]:
import yaml

def objective(trial):
    # Suggest hyperparameters
    hyp = {
        "lr0": trial.suggest_float('lr0', 1e-5, 1e-1, log=True),
        "momentum": trial.suggest_float('momentum', 0.80, 0.99),
        "weight_decay": trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True),
        "box": trial.suggest_float('box', 0.02, 0.4),
        "cls": trial.suggest_float('cls', 0.2, 1.0),
        "hsv_h": trial.suggest_float('hsv_h', 0.0, 0.1),
        "hsv_s": trial.suggest_float('hsv_s', 0.0, 0.7),
        "hsv_v": trial.suggest_float('hsv_v', 0.0, 0.4),
    }

    # Save hyp file
    hyp_path = f"trial_{trial.number}_hyp.yaml"
    with open(hyp_path, 'w') as f:
        yaml.dump(hyp, f)

    try:
        model = YOLO("yolovn.pt")
        results = model.train(
            data="/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData/data.yaml",
            epochs=100,
            imgsz=640,
            batch=16,
            name=f"trial_{trial.number}",
            cfg=hyp_path  # ✅ override training config with custom hyp
        )

        metrics = getattr(model, "metrics", None)
        if metrics and isinstance(metrics, dict):
            return metrics.get("metrics/mAP50", 0.0)
        else:
            return 0.0

    except Exception as e:
        print(f"⚠️ Trial {trial.number} failed: {e}")
        return 0.0

In [ ]:
study = optuna.create_study(direction="maximize", study_name="YOLOv8_Tuning")
study.optimize(objective, n_trials=5, timeout=60*60*3)  # try 5 trials or 3 hour

In [ ]:
best_trial = study.best_trial
best_trial_number = best_trial.number
best_model_path = f"runs/detect/trial_{best_trial.number}/weights/best.pt"

In [ ]:
best_model = YOLO(best_model_path)
val_results = best_model.val(data="/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/BulkNormalizedAnnotatedData/data.yaml", conf=0.25)

In [ ]:
print(f"Best trial number: {best_trial_number}")
print(f"Best model path: {best_model_path}")

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # go one level up

from utils import normalizer, reset, validation 
from ultralytics import YOLO

In [ ]:
#replace with best or segment
selection = "best"

In [ ]:
model = YOLO(f"runs/detect/afif_{selection}/weights/best.pt")

In [ ]:
subset_list = ["Sparse", "Normal", "Clumpped"]

for subset in subset_list:
    results = model.predict(
        source=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/{subset}/images/val",
        save=True,
        save_txt=True,
        save_conf=True,
        project=f"runs/detect/afif_{selection}",
        name=f"predict{subset}",
        exist_ok=True
    )

In [ ]:

subset_list = ["Sparse", "Normal", "Clumpped"]

for subset in subset_list:
    validation.compare_annotation_vs_prediction(
        gt_label_folder=f"runs/detect/afif_{selection}/predict{subset}/actual_labels",  # Ground truth
        pred_label_folder=f"runs/detect/afif_{selection}/predict{subset}/labels",  # prediction
        output_img_path=f"{selection}_loss_ratio_plot_{subset}.png"
    )

In [ ]:
# from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# ==== KONFIGURASI ====
model_path = "best.pt"
subset_paths = {
    "Sparse": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Sparse",
    "Normal": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Normal",
    "Clumpped": "/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/Clumpped"
}
yaml_paths = {k: os.path.join(v, "data.yaml") for k, v in subset_paths.items()}

summary_data = []
roc_data = []

for subset_name in subset_paths.keys():
    print(f"🔍 Evaluating: {subset_name}")
    
    # Run model validation
    results = model.val(data=yaml_paths[subset_name], split=f"val", save=False)

   

In [ ]:
subset_list = ["Sparse", "Normal", "Clumpped"]

for subset in subset_list:
    print(f"\n🚀 Processing subset: {subset}")
    validation.run_roc_analysis(
        yaml_path=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/{subset}/data.yaml",
        gt_folder=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/data/{subset}/labels/val",
        pred_folder=f"/Users/ahmadfariz/Projects/pythonAI/MARROWS/notebooks/runs/detect/afif_{selection}/predict{subset}/labels",
        subset_name=subset,
        selection=selection,
    )